# 1. Import das bibliotecas necessárias

In [3]:
%pip install scipy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import joblib


Note: you may need to restart the kernel to use updated packages.


# 2. Carregamento dos dados

In [5]:
# Carregando o dataset para o repositório

path = '../data/raw/Telco_customer_churn.xlsx'

df = pd.read_excel(path)

In [6]:
df.rename(columns={
            'CustomerID': 'customer_id',
            'Count': 'count',	
            'Country': 'country',	
            'State': 'state',
            'City': 'city',
            'Zip Code': 'zip_code',
            'Lat Long': 'lat_long',
            'Latitude': 'latitude',
            'Longitude': 'longitude',	
            'Gender': 'gender',
            'Senior Citizen': 'senior_citizen',
            'Partner': 'partner',	
            'Dependents': 'dependents',	
            'Tenure Months': 'tenure_months',	
            'Phone Service': 'phone_service',	
            'Multiple Lines': 'multiple_lines',	
            'Internet Service': 'internet_service',	
            'Online Security': 'online_security',	
            'Online Backup': 'online_backup',	
            'Device Protection': 'device_protection',	
            'Tech Support': 'tech_support',	
            'Streaming TV': 'streaming_tv',	
            'Streaming Movies': 'streaming_movies',	
            'Contract': 'contract',	
            'Paperless Billing': 'paperless_billing',	
            'Payment Method': 'payment_method',
            'Monthly Charges': 'monthly_charges',	
            'Total Charges': 'total_charges',	
            'Churn Label': 'churn_label',	
            'Churn Value': 'churn_value',	
            'Churn Score': 'churn_score',	
            'CLTV': 'cltv',	
            'Churn Reason': 'churn_reason'},
            inplace=True
)

# Tratamento de One hot encoding

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        7043 non-null   object 
 1   count              7043 non-null   int64  
 2   country            7043 non-null   object 
 3   state              7043 non-null   object 
 4   city               7043 non-null   object 
 5   zip_code           7043 non-null   int64  
 6   lat_long           7043 non-null   object 
 7   latitude           7043 non-null   float64
 8   longitude          7043 non-null   float64
 9   gender             7043 non-null   object 
 10  senior_citizen     7043 non-null   object 
 11  partner            7043 non-null   object 
 12  dependents         7043 non-null   object 
 13  tenure_months      7043 non-null   int64  
 14  phone_service      7043 non-null   object 
 15  multiple_lines     7043 non-null   object 
 16  internet_service   7043 

In [8]:
#categorical_cols = columns=['country','state','city','lat_long','gender','senior_citizen','partner','dependents','phone_service','multiple_lines','internet_service','device_protection','online_security','online_backup']
categorical_cols = df.select_dtypes(include='object').columns
df_encoded = pd.get_dummies(df,columns=categorical_cols, drop_first=True)

print(f"Shape pós enconding: {df_encoded.shape}")
df_encoded.head()

Shape pós enconding: (7043, 16407)


,count,zip_code,latitude,longitude,tenure_months,monthly_charges,churn_value,churn_score,cltv,customer_id_0003-MKNFE,...,churn_reason_Lack of self-service on Website,churn_reason_Limited range of services,churn_reason_Long distance charges,churn_reason_Moved,churn_reason_Network reliability,churn_reason_Poor expertise of online support,churn_reason_Poor expertise of phone support,churn_reason_Price too high,churn_reason_Product dissatisfaction,churn_reason_Service dissatisfaction
0,1,90003,33.964131,-118.272783,2,53.85,1,86,3239,False,...,False,False,False,False,False,False,False,False,False,False
1,1,90005,34.059281,-118.307420,2,70.70,1,67,2701,False,...,False,False,False,True,False,False,False,False,False,False
2,1,90006,34.048013,-118.293953,8,99.65,1,86,5372,False,...,False,False,False,True,False,False,False,False,False,False
3,1,90010,34.062125,-118.315709,28,104.80,1,84,5003,False,...,False,False,False,True,False,False,False,False,False,False
4,1,90015,34.039224,-118.266293,49,103.70,1,89,5340,False,...,False,False,False,False,False,False,False,False,False,False


# Experimentação e MVP (regressão logística - baseline)

In [ ]:
from sklearn.model_selection import train_test_split

# Divida o dataset em features (X) e target (y)
X = df_encoded.drop(columns=['target'])
y = df_encoded['target']

# Divida em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
import mlflow
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

#mlflow.set_experiment("telco_customer_churn")

#with mlflow.start_run(run_name="telco_customer_churn"):
    model = LogisticRegression(random_state=42)
    model.fit(X_train, y_train)

    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    train_accuracy = accuracy_score(y_train, y_pred_train)
    test_accuracy = accuracy_score(y_test, y_pred_test)
    test_f1 = f1_score(y_test, y_pred_test)
    test_precision = precision_score(y_test, y_pred_test)
    test_recall = recall_score(y_test, y_pred_test)

    mlflow.log_metric("train_accuracy", train_accuracy)
    mlflow.log_metric("test_accuracy", test_accuracy)
    mlflow.log_metric("test_f1_score", test_f1)
    mlflow.log_metric("test_precision", test_precision)
    mlflow.log_metric("test_recall", test_recall)

    overfitting = train_accuracy - test_accuracy
    mlflow.log_metric("overfitting", overfitting)

    mlflow.sklearn.log_model(model, "model")

    print(f"=== LOGISTIC REGRESSION ===")
    print(f"Train Accuracy: {train_accuracy:.4f}")
    print(f"Test Accuracy:  {test_accuracy:.4f}")
    print(f"Test F1 Score:  {test_f1:.4f}")

    print(f"Test Precision: {test_precision:.4f}")
    print(f"Test Recall:    {test_recall:.4f}")
    print(f"Overfitting:    {overfitting:.4f}")

# Persistir o dataframe pré-processado

In [ ]:
df_encoded.to_csv("../data/Telco_customer_cuhrn_preprocessed.csv", index=False)

In [ ]:
# Persistir o modelo treinado com MLFlow

# Persistir o modelo usando joblib
import joblib
joblib.dump(model, "../models/baseline_model.joblib")